In [1]:
from rsm3d.rsm3d import RSMBuilder  # <-- 4-circle xrayutilities builder
spec_file = '/Users/xiaogangyang/BNL.GOV Dropbox/Xiaogang Yang/isr_rsm3d/setup_6oct23'
tiff_dir  = '/Users/xiaogangyang/BNL.GOV Dropbox/Xiaogang Yang/isr_rsm3d/data_6oct23_tiff'
scan_list = (17, 18, 19, 20, 21)     # any list/tuple of scan numbers
out_vtr   = '/Users/xiaogangyang/BNL.GOV Dropbox/Xiaogang Yang/isr_rsm3d/rsm_hkl.vtr'    # output file

# Build with 4-circle (ZXZ: φ(Z) → χ(X) → ω(Z))
builder = RSMBuilder(
    spec_file, tiff_dir,
    selected_scans=scan_list,
    ub_includes_2pi=True,        # set False if your UB is "no-2π"
    center_is_one_based=False,   # True if SPEC xcenter/ycenter are 1-based
    fourc_mode="ZXZ",            # or "ZYX" if your instrument uses Z-Y-X
    motor_map={"omega":"th", "chi":"chi", "phi":"phi"},  # map to your SPEC columns
)

# Compute per-pixel Q & HKL
Q_samp, hkl, intensity = builder.compute_full()
import tifffile
import numpy as np
tifffile.imwrite('/Users/xiaogangyang/BNL.GOV Dropbox/Xiaogang Yang/isr_rsm3d/intensity.tiff', intensity.astype(np.float32))
# Regrid with xrayutilities gridder (mean or sum)
grid, (xax, yax, zax) = builder.regrid_xu(
    space="hkl",                 # "hkl" or "q"
    grid_shape=(200, 200, 200),  # adjust to taste / memory
    ranges=None,                 # or ((xmin,xmax),(ymin,ymax),(zmin,zmax)) to lock axes
    fuzzy=False,                 # True → FuzzyGridder3D; add width=... for footprint
    normalize="mean",            # "mean" or "sum"
    stream=True                  # frame-by-frame accumulation (RAM friendly)
)
grid.shape, len(xax), len(yax), len(zax)


((200, 200, 200), 200, 200, 200)

In [ ]:
from rsm3d.data_viz import RSMNapariViewer

# after compute_full & regrid_xu:
viewer = RSMNapariViewer(grid, (xax,yax,zax), raw_intensity=builder.intensity)
viewer(display_3d=True, use_log=True)

Viewer(camera=Camera(center=(np.float64(0.6176714897155762), np.float64(3.2585725784301767), np.float64(2.270174503326418)), zoom=np.float64(20.246117478503688), angles=(0.0, 0.0, 90.0), perspective=0.0, mouse_pan=True, mouse_zoom=True, orientation=(<DepthAxisOrientation.TOWARDS: 'towards'>, <VerticalAxisOrientation.DOWN: 'down'>, <HorizontalAxisOrientation.RIGHT: 'right'>)), cursor=Cursor(position=(0.0, 0.0, 0.0), viewbox=None, scaled=True, style=<CursorStyle.STANDARD: 'standard'>, size=1.0), dims=Dims(ndim=3, ndisplay=3, order=(0, 1, 2), axis_labels=('0', '1', '2'), rollable=(True, True, True), range=(RangeTuple(start=np.float64(-11.736214637756348), stop=np.float64(16.27656364440918), step=np.float64(0.11050682930491078)), RangeTuple(start=np.float64(-5.43487548828125), stop=np.float64(11.952020645141602), step=np.float64(0.08737133735388368)), RangeTuple(start=np.float64(-11.736214637756348), stop=np.float64(16.27656364440918), step=np.float64(0.1407677300611333))), margin_left=(0.

Traceback (most recent call last):
  File "/Users/xiaogangyang/pyprojects/pyisr/.pixi/envs/default/lib/python3.11/site-packages/superqt/utils/_throttler.py", line 286, in _set_future_result
    result = self._func(*self._args[: self._max_args], **self._kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/xiaogangyang/pyprojects/pyisr/.pixi/envs/default/lib/python3.11/site-packages/superqt/utils/_throttler.py", line 226, in weak_func
    return method(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/xiaogangyang/pyprojects/pyisr/.pixi/envs/default/lib/python3.11/site-packages/napari/_vispy/canvas.py", line 550, in _on_mouse_move
    self._process_mouse_event(mouse_move_callbacks, event)
  File "/Users/xiaogangyang/pyprojects/pyisr/.pixi/envs/default/lib/python3.11/site-packages/napari/_vispy/canvas.py", line 505, in _process_mouse_event
    mouse_callbacks(self.viewer, read_only_event)
  File "/Users/xiaogangyang/pyprojects/pyis

In [12]:
# Save regridded volume to a .vtr file
import vtk
from vtk.util.numpy_support import numpy_to_vtk

# grid, (xax, yax, zax) already from:
#    grid, (xax, yax, zax) = builder.regrid_xu(...)

# out_vtr is your target filename, e.g.
#    out_vtr = '/path/to/rsm_hkl.vtr'

# Build a RectilinearGrid
nx, ny, nz = len(xax), len(yax), len(zax)
rgrid = vtk.vtkRectilinearGrid()
rgrid.SetDimensions(nx, ny, nz)

# Attach coordinate arrays
coords_x = numpy_to_vtk(xax, deep=True)
coords_y = numpy_to_vtk(yax, deep=True)
coords_z = numpy_to_vtk(zax, deep=True)
rgrid.SetXCoordinates(coords_x)
rgrid.SetYCoordinates(coords_y)
rgrid.SetZCoordinates(coords_z)

# Flatten the scalar data and add as point data
flat_int = grid.flatten(order='F')
vtk_data = numpy_to_vtk(flat_int, deep=True, array_type=vtk.VTK_FLOAT)
vtk_data.SetName('intensity')
rgrid.GetPointData().SetScalars(vtk_data)

# Write out the file
writer = vtk.vtkXMLRectilinearGridWriter()
writer.SetFileName(out_vtr)
writer.SetInputData(rgrid)
writer.Write()

print(f"Regridded volume saved to {out_vtr}")

import vtk
from vtk.util.numpy_support import numpy_to_vtk

# grid, (xax, yax, zax) already from:
#    grid, (xax, yax, zax) = builder.regrid_xu(...)

# out_vtr is your target filename, e.g.
#    out_vtr = '/path/to/rsm_hkl.vtr'

# Build a RectilinearGrid
nx, ny, nz = len(xax), len(yax), len(zax)
rgrid = vtk.vtkRectilinearGrid()
rgrid.SetDimensions(nx, ny, nz)

# Attach coordinate arrays
coords_x = numpy_to_vtk(xax, deep=True)
coords_y = numpy_to_vtk(yax, deep=True)
coords_z = numpy_to_vtk(zax, deep=True)
rgrid.SetXCoordinates(coords_x)
rgrid.SetYCoordinates(coords_y)
rgrid.SetZCoordinates(coords_z)

# Flatten the scalar data and add as point data
flat_int = grid.flatten(order='F')
vtk_data = numpy_to_vtk(flat_int, deep=True, array_type=vtk.VTK_FLOAT)
vtk_data.SetName('intensity')
rgrid.GetPointData().SetScalars(vtk_data)

# Write out the file
writer = vtk.vtkXMLRectilinearGridWriter()
writer.SetFileName(out_vtr)
writer.SetInputData(rgrid)
writer.Write()

print(f"Regridded volume saved to {out_vtr}")

Regridded volume saved to /Users/xiaogangyang/BNL.GOV Dropbox/Xiaogang Yang/isr_rsm3d/rsm_hkl.vtr
Regridded volume saved to /Users/xiaogangyang/BNL.GOV Dropbox/Xiaogang Yang/isr_rsm3d/rsm_hkl.vtr


In [22]:
import numpy as np
import napari

# ---------- helpers ----------
def volume_from_grid_axes(grid, axes):
    """
    (nx,ny,nz) + 1D axes -> (nz,ny,nx) volume plus (scale, translate) for napari.
    Uses average spacing for the linear world transform; exact coords come from axes below.
    """
    xax, yax, zax = [np.asarray(a) for a in axes]
    nx, ny, nz = len(xax), len(yax), len(zax)
    if grid.shape != (nx, ny, nz):
        raise ValueError(f"grid shape {grid.shape} != ({nx},{ny},{nz}) from axes")

    vol = grid.transpose(2, 1, 0).copy()  # (Z,Y,X)

    def _avg_step(a): return float(np.diff(a).mean()) if len(a) > 1 else 1.0
    dx, dy, dz = _avg_step(xax), _avg_step(yax), _avg_step(zax)

    translate = (float(zax[0]), float(yax[0]), float(xax[0]))  # (Z,Y,X)
    scale     = (dz, dy, dx)
    is_uniform = (
        np.allclose(np.diff(xax), dx, rtol=1e-5, atol=1e-8) and
        np.allclose(np.diff(yax), dy, rtol=1e-5, atol=1e-8) and
        np.allclose(np.diff(zax), dz, rtol=1e-5, atol=1e-8)
    )
    return vol, scale, translate, is_uniform

def log1p_clip(a):
    a = np.asarray(a)
    return np.log1p(np.maximum(a, 0.0))

def index_to_axis_value(ax, idx):
    """
    Map a (possibly fractional) data index -> exact world coordinate using your axis array.
    Works for non-uniform axes.
    """
    n = len(ax)
    if n == 0:
        return np.nan
    if idx <= 0:
        return float(ax[0])
    if idx >= n - 1:
        return float(ax[-1])
    i0 = int(np.floor(idx))
    t  = float(idx - i0)
    return float((1.0 - t) * ax[i0] + t * ax[i0 + 1])

# ---------- build napari viewer ----------
# Assumes you already have: grid, (xax, yax, zax)
volume, scale, translate, is_uniform = volume_from_grid_axes(grid, (xax, yax, zax))

v = napari.Viewer(ndisplay=3, title="RSM viewer")

vol_log = log1p_clip(volume)
lo, hi = np.percentile(vol_log, [1, 99.8])  # quick sensible contrast
img_layer = v.add_image(
    vol_log,
    name="RSM (log1p)",
    colormap="viridis",
    rendering="attenuated_mip",
    blending="translucent",
    opacity=1.0,
    scale=scale,          # linear world transform (approx if non-uniform axes)
    translate=translate,
    # contrast_limits=(float(lo), float(hi)),
)

# UI niceties
v.axes.visible = True
v.axes.colored = True
v.axes.arrows = True
v.scale_bar.visible = True
v.scale_bar.unit = ""              # set to "Å⁻¹" if you’re viewing Q-space
v.dims.axis_labels = ("L", "K", "H")  # for HKL; use ("Qz","Qy","Qx") for Q-space

# ---------- super-thin corners & outline (world coordinates) ----------
zmin, zmax = float(zax[0]), float(zax[-1])
ymin, ymax = float(yax[0]), float(yax[-1])
xmin, xmax = float(xax[0]), float(xax[-1])

corners_world = np.array([
    [zmin, ymin, xmin],
    [zmin, ymin, xmax],
    [zmin, ymax, xmin],
    [zmin, ymax, xmax],
    [zmax, ymin, xmin],
    [zmax, ymin, xmax],
    [zmax, ymax, xmin],
    [zmax, ymax, xmax],
], dtype=float)

# tiny corner markers: << 1 voxel (use isotropic size for napari stable)
voxel = np.array(scale, dtype=float)           # (dz, dy, dx)
corner_size = min(voxel) * 0.15                # 0.15 of smallest voxel size
corner_sizes = np.full(8, corner_size)         # (8,) isotropic marker size
v.add_points(
    corners_world,
    name="Outline corners",
    size=corner_sizes,      # isotropic (length 8)
    face_color="red",
    opacity=0.9,
    blending="additive",
)

# hairline outline
box_edges = np.array([
    [0,1],[0,2],[0,4],
    [1,3],[1,5],
    [2,3],[2,6],
    [3,7],
    [4,5],[4,6],
    [5,7],
    [6,7],
], dtype=int)
edge_segments = [corners_world[e] for e in box_edges]
v.add_shapes(
    edge_segments,
    shape_type="line",
    edge_color="yellow",
    edge_width=0.05,               # sub-pixel hairline
    opacity=0.9,
    blending="additive",
    name="Outline box",
)

# slim world-axis vectors
Lx, Ly, Lz = xmax - xmin, ymax - ymin, zmax - zmin
axes_len = 0.10 * max(Lx, Ly, Lz)
origin = np.array([zmin, ymin, xmin], dtype=float)
vectors = np.stack([
    np.vstack([origin, origin + np.array([axes_len, 0, 0])]),  # +Z
    np.vstack([origin, origin + np.array([0, axes_len, 0])]),  # +Y
    np.vstack([origin, origin + np.array([0, 0, axes_len])]),  # +X
], axis=0)
# v.add_vectors(
#     vectors,
#     name="World axes",
#     edge_color=["cyan", "lime", "magenta"],
#     edge_width=0.75,
#     blending="translucent_no_depth",
# )

# ---------- live coordinate HUD (H, K, L + intensity under cursor) ----------
# Uses exact coordinate mapping from axis arrays (works for non-uniform axes).
# If napari version has viewer.text_overlay, we’ll use it; otherwise we print to console.
def on_mouse_move(viewer, event):
    pos_world = viewer.cursor.position
    if pos_world is None:
        return
    # data indices in (Z,Y,X) for this layer
    zi, yi, xi = img_layer.world_to_data(pos_world)
    # intensity sample (linear intensity from 'volume')
    I = np.nan
    zi_i, yi_i, xi_i = int(np.round(zi)), int(np.round(yi)), int(np.round(xi))
    if (0 <= zi_i < volume.shape[0]) and (0 <= yi_i < volume.shape[1]) and (0 <= xi_i < volume.shape[2]):
        I = float(volume[zi_i, yi_i, xi_i])

    # exact HKL (or Q) using axis arrays
    H = index_to_axis_value(xax, xi)
    K = index_to_axis_value(yax, yi)
    L = index_to_axis_value(zax, zi)
    text = f"H={H:.4f}   K={K:.4f}   L={L:.4f}    I={I:.3g}"

    if hasattr(viewer, "text_overlay") and viewer.text_overlay is not None:
        overlay = viewer.text_overlay
        overlay.visible = True
        overlay.position = 'top_left'
        overlay.color = 'white'
        overlay.font_size = 12
        overlay.text = text
    else:
        print(text, end="\r")

v.mouse_move_callbacks.append(on_mouse_move)

# Optional: press 'C' to toggle the HUD
@v.bind_key('C')
def _toggle_coords(viewer):
    if hasattr(viewer, "text_overlay") and viewer.text_overlay is not None:
        viewer.text_overlay.visible = not viewer.text_overlay.visible

print("Uniform voxel spacing (for the linear transform):", is_uniform)
v

Uniform voxel spacing (for the linear transform): True


Viewer(camera=Camera(center=(np.float64(0.6176714897155762), np.float64(3.2585725784301767), np.float64(2.270174503326418)), zoom=np.float64(20.246117478503688), angles=(0.0, 0.0, 90.0), perspective=0.0, mouse_pan=True, mouse_zoom=True, orientation=(<DepthAxisOrientation.TOWARDS: 'towards'>, <VerticalAxisOrientation.DOWN: 'down'>, <HorizontalAxisOrientation.RIGHT: 'right'>)), cursor=Cursor(position=(0.0, 0.0, 0.0), viewbox=None, scaled=True, style=<CursorStyle.STANDARD: 'standard'>, size=1.0), dims=Dims(ndim=3, ndisplay=3, order=(0, 1, 2), axis_labels=('L', 'K', 'H'), rollable=(True, True, True), range=(RangeTuple(start=np.float64(-10.377758026123047), stop=np.float64(11.6131010055542), step=np.float64(0.11050682930491078)), RangeTuple(start=np.float64(-5.43487548828125), stop=np.float64(11.952020645141602), step=np.float64(0.08737133735388368)), RangeTuple(start=np.float64(-11.736214637756348), stop=np.float64(16.27656364440918), step=np.float64(0.1407677300611333))), margin_left=(0.0

In [1]:
import numpy as np
import napari

# ---------- helpers ----------
def volume_from_grid_axes(grid, axes):
    """
    (nx,ny,nz) + 1D axes -> (nz,ny,nx) volume and (scale, translate) for napari.
    Uses average spacing for the linear world transform (exact coords still come from axes).
    """
    xax, yax, zax = [np.asarray(a) for a in axes]
    nx, ny, nz = len(xax), len(yax), len(zax)
    if grid.shape != (nx, ny, nz):
        raise ValueError(f"grid shape {grid.shape} != ({nx},{ny},{nz}) from axes")

    vol = grid.transpose(2, 1, 0).copy()  # (Z,Y,X)

    def _avg_step(a): return float(np.diff(a).mean()) if len(a) > 1 else 1.0
    dx, dy, dz = _avg_step(xax), _avg_step(yax), _avg_step(zax)

    translate = (float(zax[0]), float(yax[0]), float(xax[0]))  # (Z,Y,X)
    scale     = (dz, dy, dx)
    is_uniform = (
        np.allclose(np.diff(xax), dx, rtol=1e-5, atol=1e-8) and
        np.allclose(np.diff(yax), dy, rtol=1e-5, atol=1e-8) and
        np.allclose(np.diff(zax), dz, rtol=1e-5, atol=1e-8)
    )
    return vol, scale, translate, is_uniform

def log1p_clip(a):
    a = np.asarray(a)
    return np.log1p(np.maximum(a, 0.0))

# ---------- build napari viewer (FORCE VOLUME MODE) ----------
# Assumes you already have: grid, (xax, yax, zax)
volume, scale, translate, is_uniform = volume_from_grid_axes(grid, (xax, yax, zax))

# Sanity: ensure truly 3-D
assert volume.ndim == 3, f"Expected 3-D, got {volume.ndim}D"
nz, ny, nx = volume.shape
assert nz > 1 and ny > 1 and nx > 1, f"One dimension is singleton: (nz,ny,nx)={volume.shape}"

v = napari.Viewer(title="RSM viewer")
# force viewer to 3-D
v.dims.ndisplay = 3

vol_log = log1p_clip(volume)
lo, hi = np.percentile(vol_log, [1, 99.8])

img_layer = v.add_image(
    vol_log,
    name="RSM (log1p)",
    scale=scale,                # (Z,Y,X)
    translate=translate,        # (Z,Y,X)
    contrast_limits=(float(lo), float(hi)),
)

# Make absolutely sure it's a VOLUME, not a slicing plane
if hasattr(img_layer, "depiction"):
    img_layer.depiction = "volume"     # napari >= 0.5
# Pick a 3-D renderer
if hasattr(img_layer, "rendering"):
    img_layer.rendering = "attenuated_mip"

# Make sure the viewer is in 3-D after adding the layer
v.dims.ndisplay = 3

# Frame the whole volume and set a non-orthogonal camera so it is clearly 3-D
try:
    v.reset_view()
    v.camera.angles = (30, 30, 0)  # yaw, pitch, roll (deg) for a nice 3-D angle
    v.camera.zoom = 1.0
except Exception:
    pass

# UI niceties
v.axes.visible = True
v.axes.colored = True
v.axes.arrows = True
v.scale_bar.visible = True
v.scale_bar.unit = ""             # "Å⁻¹" if Q-space
v.dims.axis_labels = ("L", "K", "H")

# ---------- draw thin outline & corners (unchanged functionality, very light) ----------
zmin, zmax = float(zax[0]), float(zax[-1])
ymin, ymax = float(yax[0]), float(yax[-1])
xmin, xmax = float(xax[0]), float(xax[-1])

corners_world = np.array([
    [zmin, ymin, xmin],
    [zmin, ymin, xmax],
    [zmin, ymax, xmin],
    [zmin, ymax, xmax],
    [zmax, ymin, xmin],
    [zmax, ymin, xmax],
    [zmax, ymax, xmin],
    [zmax, ymax, xmax],
], dtype=float)

voxel = np.array(scale, dtype=float)           # (dz, dy, dx)
corner_size = float(min(voxel) * 0.15)
v.add_points(
    corners_world,
    name="Outline corners",
    size=np.full(8, corner_size),  # isotropic markers
    face_color="red",
    # edge_width=0,
    opacity=0.9,
    blending="additive",
)

box_edges = np.array([
    [0,1],[0,2],[0,4],
    [1,3],[1,5],
    [2,3],[2,6],
    [3,7],
    [4,5],[4,6],
    [5,7],
    [6,7],
], dtype=int)
edge_segments = [corners_world[e] for e in box_edges]
v.add_shapes(
    edge_segments,
    shape_type="line",
    edge_color="yellow",
    edge_width=0.1,               # pixel units (hairline)
    # edge_width_is_relative=False,
    opacity=0.9,
    blending="additive",
    name="Outline box",
)

# ---------- diagnostics: confirm world extent covers the whole volume ----------
try:
    # data extent (indices) and world extent (coords)
    print("Data extent (Z,Y,X):", img_layer.extent.data)    # ( (zmin,zmax), (ymin,ymax), (xmin,xmax) ) in index space
    print("World extent (Z,Y,X):", img_layer.extent.world)  # same, transformed by scale/translate
except Exception:
    pass

print("Volume shape (Z,Y,X):", volume.shape)
print("Uniform voxel spacing (for linear transform):", is_uniform)

v

NameError: name 'grid' is not defined